# Petit modèle génératif de chiffres — diffusion / flow matching

Par Pascal Montmory pour ENINCA CONSULTING. Prototype contrôlé sur CPU ; compilation native et performances TPU à vérifier. Aucun gain commercial n’est garanti. La reproductibilité exacte concerne les entiers, pas les transformations flottantes ou un entraînement complet. Le pilote court est activé par défaut.

Choisir **Exécution > Modifier le type d’environnement > TPU**. L’allocation dépend de votre compte et de ses quotas. Aucun identifiant GitHub ni envoi automatique des résultats.

## 1. Dépendances et accès TPU
Installe les versions épinglées et vérifie un vrai TPU. En cas d’échec, télécharger le diagnostic avec la cellule 4 ; aucun repli CPU.

In [ ]:
import datetime, importlib.metadata, os, subprocess, sys
from pathlib import Path
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PREFLIGHT_LOG = Path.cwd() / ("applications-tpu-preflight-" + stamp + ".log")
TPU_READY = False
def logged(command, env=None):
    with PREFLIGHT_LOG.open("a") as log:
        with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True); log.write(line); log.flush()
                code = process.wait()
            except BaseException:
                process.terminate(); process.wait(); raise
    if code:
        raise RuntimeError("Setup / TPU access failed. Download diagnostics in cell 4. No CPU fallback.")
pins = {"jax":"0.10.2", "jaxlib":"0.10.2", "numpy":"2.4.6", "scipy":"1.17.1", "scikit-learn":"1.7.2", "matplotlib":"3.10.7"}
try:
    ready = all(importlib.metadata.version(k) == v for k,v in pins.items()) and bool(importlib.metadata.version("libtpu"))
except importlib.metadata.PackageNotFoundError:
    ready = False
if not ready:
    logged([sys.executable,"-m","pip","install","--upgrade","jax[tpu]==0.10.2"] + [k+"=="+v for k,v in pins.items() if k!="jax"])
logged([sys.executable,"-u","-c","import jax; assert jax.default_backend() == 'tpu'; print(jax.__version__, jax.devices())"], env={**os.environ,"JAX_PLATFORMS":"tpu"})
TPU_READY = True


## 2. Sources et empreintes
Kit autonome intégré au notebook, avec contrôles SHA-256.

In [ ]:
import base64, hashlib, io, json, tarfile
from pathlib import Path
import datetime
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '70db71817e61021d5185f3e19e3e75beb9258e409f015872a053c1d5f0bd6626'
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
KIT = Path("/content") / ("eninca-applications-" + stamp)
KIT.mkdir(exist_ok=False)
with tarfile.open(fileobj=io.BytesIO(payload),mode="r:gz") as tar:
    seen = set()
    for member in tar.getmembers():
        target = (KIT / member.name).resolve()
        if not member.isfile() or not target.is_relative_to(KIT.resolve()) or member.name in seen:
            raise ValueError("Invalid archive member")
        seen.add(member.name)
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(tar.extractfile(member).read())
manifest = json.loads((KIT / "manifest.json").read_text())
assert seen == set(manifest) | {"manifest.json"}
for rel,digest in manifest.items():
    assert hashlib.sha256((KIT / rel).read_bytes()).hexdigest() == digest, rel
print("Source hashes PASS:", len(manifest))


## 3. Vérifications puis expérience
Les compilations précèdent les chronométrages. Garder QUICK=True pour le premier essai. Le protocole complet utilise les paramètres définis avant les mesures. Les durées CPU ne prédisent pas les durées TPU.

In [ ]:
import subprocess, sys
if not globals().get("TPU_READY",False):
    raise RuntimeError("Cell 1 must succeed before running a TPU benchmark.")
QUICK = True  # False: longer, prespecified protocol; not an optimization sweep.
RESULTS = KIT / "results" / "diffusion-tpu"
RESULTS.parent.mkdir(exist_ok=True)
command = [sys.executable,"-u",str(KIT / "applications/diffusion/run.py"),"--backend","tpu","--out",str(RESULTS)]
if QUICK: command.append("--quick")
with (KIT / "run.log").open("w") as log:
    with subprocess.Popen(command,cwd=KIT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as process:
        try:
            for line in process.stdout:
                print(line,end="",flush=True); log.write(line); log.flush()
            code = process.wait()
        except BaseException:
            process.terminate(); process.wait(); raise
print("Results:",RESULTS)
if code: raise RuntimeError("Experiment failed. Download diagnostics using cell 4.")
from IPython.display import display, Image
for name in ("learning.png","samples.png","error-vs-time.png"):
    if (RESULTS / name).exists(): display(Image(filename=str(RESULTS / name)))


## 4. Télécharger résultats ou diagnostics
Utiliser aussi après un échec de la première ou de la troisième cellule.

In [ ]:
import tarfile
from google.colab import files
log = globals().get("PREFLIGHT_LOG")
results = globals().get("RESULTS")
if not globals().get("TPU_READY",False) and log is not None and log.exists():
    files.download(str(log))
elif results is not None and results.exists():
    archive = results.with_suffix(".tar.gz")
    with tarfile.open(archive,"w:gz") as tar:
        tar.add(results,arcname=results.name,filter=lambda i: None if i.name.endswith((".so",".dylib")) else i)
        for path in (KIT / "run.log", KIT / "manifest.json", log):
            if path is not None and path.exists(): tar.add(path,arcname=path.name)
    files.download(str(archive))
elif log is not None and log.exists():
    files.download(str(log))
else:
    raise RuntimeError("No diagnostics yet; start with cell 1.")
